# Decision Tree

Learn this model in order: **setup → baseline → initial model → tuning → interpretation**.

We keep all 395 students, including 38 recorded zero grades. The same 316 training students and five folds are used in every notebook. The 79 test students belong to final evaluation in Notebook 04.

There are two clearly separated scenarios: without G1/G2 (main) and with both grades (later stage). Input timing, particularly absences, still needs verification. MAE remains the selection metric. The test results have already been inspected: rerunning the unchanged experiment reproduces it, rather than creating a new unseen test.

Results and explanations stay in this notebook. Only fitted pipelines are saved to `models/` for the comparison and evaluation notebooks.

## 1 Setup

Shared helpers in `src/model_utils.py` handle the split and repeated metric calculations. Model settings, fitting and tuning remain visible here. Preprocessing is always fitted inside the training fold.

MAE/RMSE measure grade-point error; R² describes squared-error performance. Accuracy, precision, recall, F1 and specificity evaluate the derived **predicted G3 < 10** flag against **actual G3 < 10**. No rounding or threshold tuning is used.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_validate, GridSearchCV

PROJECT_ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "data/student-mat.csv").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from model_utils import (
    load_data, model_pipeline, preprocessing_for,
    CV, SCORING, METRICS, fold_results, cv_summary, pooled_risk,
)

df, train_ids, test_ids = load_data(PROJECT_ROOT)
y_train = df.loc[train_ids, "G3"]
X_no_grades = df.loc[train_ids].drop(columns=["G1", "G2", "G3"])
X_with_grades = df.loc[train_ids].drop(columns="G3")
print("Training:", len(train_ids), "| Held-out test:", len(test_ids))
print("Inputs without / with G1,G2:", X_no_grades.shape[1], "/", X_with_grades.shape[1])

Training: 316 | Held-out test: 79
Inputs without / with G1,G2: 30 / 32


In [2]:
MODEL_NAME = 'Decision Tree'
MODEL_SLUG = 'decision_tree'
SCALE_NUMERIC = False

from sklearn.tree import DecisionTreeRegressor
INITIAL_ESTIMATOR = DecisionTreeRegressor(max_depth=3, min_samples_leaf=10, random_state=42)
MODEL_PARAM_GRID = {'model__max_depth': [2, 3, 5, None], 'model__min_samples_leaf': [2, 5, 10]}

## 2 Constant baselines

Mean and median baselines learn their constant from each fold's training labels. They ignore the inputs, so one set of scores applies to both scenarios. Precision is undefined when nobody is flagged; missing values are left visible rather than replaced with perfect scores.

In [3]:
baseline_folds = {}
for strategy in ["mean", "median"]:
    scores = cross_validate(DummyRegressor(strategy=strategy), X_no_grades, y_train,
        cv=CV, scoring=SCORING, error_score="raise")
    baseline_folds[strategy] = fold_results(scores)

display(pd.DataFrame({name: folds[METRICS].mean()
    for name, folds in baseline_folds.items()}).T.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.4029,4.5635,-0.0350,0.6744,NaN,0.0,0.0,1.0
median,3.3403,4.5931,-0.0519,0.6744,NaN,0.0,0.0,1.0


## 3 Without G1 and G2

This is the main prediction scenario. Excluding grades does not by itself prove that all other inputs are available early.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [4]:
initial_no_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=False)
initial_scores_no_grades = cross_validate(initial_no_grades, X_no_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_no_grades = fold_results(initial_scores_no_grades)
display(cv_summary(initial_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.1304,4.1762,0.1033,0.6459,0.4710,0.3572,0.3929,0.7669
std,0.2425,0.3071,0.1878,0.1086,0.1999,0.1444,0.1512,0.1576
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [5]:
grid_no_grades = dict(MODEL_PARAM_GRID)
grid_no_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=False)
    for use_log in [False, True]
]
search_no_grades = GridSearchCV(initial_no_grades, grid_no_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_no_grades.fit(X_no_grades, y_train)
best_no_grades = search_no_grades.best_estimator_
tuned_folds_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_)

print("Model settings:", best_no_grades.named_steps["model"])
print("Log absences:", best_no_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_no_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,3.0791,4.0636,0.1549,0.6806,0.7142,0.3153,0.3561,0.8605
std,0.3238,0.2893,0.1607,0.0895,0.3132,0.2300,0.1645,0.1857
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: DecisionTreeRegressor(max_depth=2, min_samples_leaf=2, random_state=42)
Log absences: False


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [6]:
comparison_no_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_no_grades[METRICS].mean(),
    "Tuned model": tuned_folds_no_grades[METRICS].mean(),
}).T
display(comparison_no_grades.round(4))

training_no_grades = fold_results(search_no_grades.cv_results_, search_no_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_no_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_no_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_no_grades),
    "Tuned model": pooled_risk(tuned_folds_no_grades)}).T.round(4))
display(tuned_folds_no_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.0000,1.0000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.0000,1.0000
Initial model,3.1304,4.1762,0.1033,0.6459,0.4710,0.3572,0.3929,0.7669
Tuned model,3.0791,4.0636,0.1549,0.6806,0.7142,0.3153,0.3561,0.8605


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.6456,0.4483,0.3786,0.4105,0.7746,39.0,48.0,64.0,165.0
Tuned model,0.6804,0.5161,0.3107,0.3879,0.8592,32.0,30.0,71.0,183.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,2.8052,3.7926,-0.0018,0.6094,1.0000,0.1071,0.1935,1.0000,36.0,0.0,25.0,3.0
1,2,3.5467,4.4262,0.3494,0.6032,0.4375,0.6667,0.5283,0.5714,24.0,18.0,7.0,14.0
2,3,2.7410,3.7454,0.0012,0.6349,0.3333,0.2778,0.3030,0.7778,35.0,10.0,13.0,5.0
3,4,3.1742,4.1382,0.1400,0.7778,1.0000,0.1250,0.2222,1.0000,47.0,0.0,14.0,2.0
4,5,3.1286,4.2155,0.2858,0.7778,0.8000,0.4000,0.5333,0.9535,41.0,2.0,12.0,8.0


Tuned training MAE: 2.8614
Tuned validation MAE: 3.0791


## 4 With G1 and G2

This is a later-stage comparison, usable only once both prior grades are known. It answers a different prediction-timing question.

### Initial model

Start with the configuration in the setup and original absences. The table shows fold means, standard deviations and how many folds have defined metrics.

In [7]:
initial_with_grades = model_pipeline(INITIAL_ESTIMATOR,
    scale_numeric=SCALE_NUMERIC, include_prior_grades=True)
initial_scores_with_grades = cross_validate(initial_with_grades, X_with_grades, y_train,
    cv=CV, scoring=SCORING, return_train_score=True, n_jobs=-1, error_score="raise")
initial_folds_with_grades = fold_results(initial_scores_with_grades)
display(cv_summary(initial_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,1.2197,1.8750,0.8165,0.8417,0.6932,0.9594,0.7964,0.7855
std,0.1796,0.2904,0.0635,0.0711,0.1440,0.0421,0.0932,0.1064
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


### Tune using training cross-validation

Search the model settings from the setup and original versus log-transformed absences. Linear Regression searches only that preprocessing choice; it has no Ridge-style alpha. Select by MAE, then refit the chosen pipeline on all 316 training students. Classification metrics are supporting evidence and do not change selection.

In [8]:
grid_with_grades = dict(MODEL_PARAM_GRID)
grid_with_grades["preprocessing"] = [
    preprocessing_for(use_log=use_log, scale_numeric=SCALE_NUMERIC,
        include_prior_grades=True)
    for use_log in [False, True]
]
search_with_grades = GridSearchCV(initial_with_grades, grid_with_grades,
    scoring=SCORING, refit="MAE", cv=CV, return_train_score=True,
    n_jobs=-1, error_score="raise")
search_with_grades.fit(X_with_grades, y_train)
best_with_grades = search_with_grades.best_estimator_
tuned_folds_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_)

print("Model settings:", best_with_grades.named_steps["model"])
print("Log absences:", best_with_grades.named_steps["preprocessing"].named_steps["absence_transform"].kw_args["use_log"])
display(cv_summary(tuned_folds_with_grades).round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
mean,1.0108,1.6618,0.8646,0.8892,0.7702,0.9273,0.8367,0.8685
std,0.2601,0.6249,0.0588,0.0253,0.0702,0.1167,0.0648,0.0390
defined folds,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000,5.0000


Model settings: DecisionTreeRegressor(max_depth=5, min_samples_leaf=5, random_state=42)
Log absences: False


### Compare and inspect errors

Compare regression and risk metrics on the same folds. The training/validation MAE gap helps identify overfitting. Pooled risk metrics come from adding the five fold confusion matrices; pooled rates can differ from fold averages. FN means missed at-risk students and FP means unnecessary flags.

The selected settings were chosen using these same folds, so their scores may be optimistic. They are not independent final-test or nested-CV estimates.

In [9]:
comparison_with_grades = pd.DataFrame({
    "Mean baseline": baseline_folds["mean"][METRICS].mean(),
    "Median baseline": baseline_folds["median"][METRICS].mean(),
    "Initial model": initial_folds_with_grades[METRICS].mean(),
    "Tuned model": tuned_folds_with_grades[METRICS].mean(),
}).T
display(comparison_with_grades.round(4))

training_with_grades = fold_results(search_with_grades.cv_results_, search_with_grades.best_index_, partition="train")
print("Tuned training MAE:", round(training_with_grades.MAE.mean(), 4))
print("Tuned validation MAE:", round(tuned_folds_with_grades.MAE.mean(), 4))
display(pd.DataFrame({"Initial model": pooled_risk(initial_folds_with_grades),
    "Tuned model": pooled_risk(tuned_folds_with_grades)}).T.round(4))
display(tuned_folds_with_grades.round(4))

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Mean baseline,3.4029,4.5635,-0.0350,0.6744,NaN,0.0000,0.0000,1.0000
Median baseline,3.3403,4.5931,-0.0519,0.6744,NaN,0.0000,0.0000,1.0000
Initial model,1.2197,1.8750,0.8165,0.8417,0.6932,0.9594,0.7964,0.7855
Tuned model,1.0108,1.6618,0.8646,0.8892,0.7702,0.9273,0.8367,0.8685


,Accuracy,Precision,Recall,F1,Specificity,TP,FP,FN,TN
Initial model,0.8418,0.6828,0.9612,0.7984,0.7840,99.0,46.0,4.0,167.0
Tuned model,0.8892,0.7742,0.9320,0.8458,0.8685,96.0,28.0,7.0,185.0


,Fold,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity,TN,FP,FN,TP
0,1,0.8574,1.2872,0.8846,0.9062,0.8438,0.9643,0.9000,0.8611,31.0,5.0,1.0,27.0
1,2,1.4168,2.4941,0.7934,0.8889,0.7500,1.0000,0.8571,0.8333,35.0,7.0,0.0,21.0
2,3,0.7668,1.1131,0.9118,0.8571,0.7647,0.7222,0.7429,0.9111,41.0,4.0,5.0,13.0
3,4,0.9000,1.2470,0.9219,0.8730,0.6667,1.0000,0.8000,0.8298,39.0,8.0,0.0,16.0
4,5,1.1131,2.1674,0.8112,0.9206,0.8261,0.9500,0.8837,0.9070,39.0,4.0,1.0,19.0


Tuned training MAE: 0.651
Tuned validation MAE: 1.0108


## 5 Compare scenarios and save the fitted models

The score difference reflects the information available at different prediction times. Do not present the G1/G2 scenario as an early prediction. Save only the two complete fitted pipelines; tables and findings remain inside the notebook.

In [10]:
display(pd.DataFrame({
    "Without G1/G2": tuned_folds_no_grades[METRICS].mean(),
    "With G1/G2": tuned_folds_with_grades[METRICS].mean(),
}).T.round(4))

model_dir = PROJECT_ROOT / "models/experiments"
model_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_no_grades, model_dir / "decision_tree__no_prior_grades.joblib")
joblib.dump(best_with_grades, model_dir / "decision_tree__with_prior_grades.joblib")
print("Saved fitted pipelines. No experiment tables or reports are exported.")

,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Specificity
Without G1/G2,3.0791,4.0636,0.1549,0.6806,0.7142,0.3153,0.3561,0.8605
With G1/G2,1.0108,1.6618,0.8646,0.8892,0.7702,0.9273,0.8367,0.8685


Saved fitted pipelines. No experiment tables or reports are exported.
